<a href="https://colab.research.google.com/github/WeiDeHuang1019/eyes-catching/blob/main/Eyes_Catching_HW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eyes-Catching 嵌入式影像處理HW1[F114112124黃維得]

## (1)find ROI part

In [ ]:
import numpy as np
import cv2
from google.colab.patches import cv2_imshow

#讀取圖片
im = cv2.imread("face4.jpg")
im_drawCircle = im.copy()
#im = cv2.convertScaleAbs(im, alpha=2, beta=0)
cv2_imshow(im)

#取得圖片寬
img_width=im.shape[1]
#dot = int(img_width/10)
print("Image Width =", img_width)

#轉灰階
img_gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
cv2_imshow(img_gray)

#二質化
A, th = cv2.threshold(img_gray, 215, 255, cv2.THRESH_BINARY_INV)
cv2_imshow(th)

#逐列掃描分析輪廓分布 ->找臉中心
H, W = th.shape
x_centers = []
rows = []
for y in range(H):
    xs = np.where(th[y] > 0)[0]  # 找這一列所有「人」的 pixel
    if len(xs) < 10:  # 太少代表不是有效列（例如雜訊）
        continue
    x_left = xs[0]
    x_right = xs[-1]
    width = x_right - x_left
    x_mid = (x_left + x_right) / 2.0
    x_centers.append(x_mid)
    rows.append((y, x_left, x_right, width, x_mid))
x_face = int(np.median(x_centers)) #臉中心(x軸)
print("Face center x =", x_face)

# 抓頭頂位置
y_top = rows[0][0] #頭頂位置(y軸)
print("y_top =", y_top)
ys = np.array([r[0] for r in rows])
widths = np.array([r[3] for r in rows])

# 抓脖子位置
y_min_search = int(H * 0.15)
y_max_search = int(H * 0.65)
candidate = [(y, w) for (y, _, _, w, _) in rows if y_min_search <= y <= y_max_search]
y_vals = np.array([c[0] for c in candidate])
w_vals = np.array([c[1] for c in candidate])
idx = np.argmin(w_vals)
y_neck = int(y_vals[idx]) #脖子位置(y軸)
print("y_neck =", y_neck)

# 定義單位長度
dot = int((y_neck - y_top)/10)
print("dot =", dot)

# 抓眼睛位置
head_h = y_neck - y_top
y_eye_center = int(y_top + 0.55 * head_h) #眼睛位置(y軸)
y1 = int(y_top + 0.25 * head_h)
y2 = int(y_top + 0.50 * head_h)
print("eye center y =", y_eye_center)
print("eye ROI y-range =", y1, y2)

# 以臉與眼睛中心，畫 ROI 範圍
x1 = max(x_face - int(dot*3), 0)
y1 = max(y_eye_center - int(dot*1.2), 0)
x2 = min(x_face + int(dot*3), im.shape[1])
y2 = min(y_eye_center + int(dot*1.2), im.shape[0])
box_width = x2 - x1
box_height = y2 - y1
roi = im[y1:y2, x1:x2].copy()
cv2.rectangle(im, (x1, y1), (x2, y2), (255, 0, 0), 2)
cv2_imshow(im)


##(2)find eyeball part

二質化粗略抓出瞳孔區塊

In [ ]:
im_drawCircle = im.copy()
cv2_imshow(roi)
# 高斯模糊
#roi = cv2.GaussianBlur(roi, (9, 9), 0)
# 轉灰階
roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
cv2_imshow(roi_gray)
# 二質化抓最黑瞳孔
th = np.percentile(roi_gray, 1) # 自適應找最黑的 5%
_, th_in_roi = cv2.threshold(roi_gray, th, 255, cv2.THRESH_BINARY_INV)
cv2_imshow(th_in_roi)
# open
th_in_roi = cv2.morphologyEx(th_in_roi, cv2.MORPH_CLOSE, np.ones((int(box_height/10), int(box_height/10)), np.uint8))
cv2_imshow(th_in_roi)

抓出兩群的群心位置

In [ ]:

# 找連通元件
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(th_in_roi, connectivity=8)

# 收集候選白色群
components = []
for i in range(1, num_labels):   # 0 是背景
    area = stats[i, cv2.CC_STAT_AREA]
    if area > 3:   # 面積門檻，可自己調
        cx, cy = centroids[i]
        components.append({
            "label": i,
            "area": area,
            "cx": cx,
            "cy": cy
        })

# 依 x 座標排序（左到右）
components = sorted(components, key=lambda c: c["cx"])

# 如果想只取兩群，可取前兩個面積最大的，或左右各一群
# 這張圖通常直接取面積最大的兩群就可以
components = sorted(components, key=lambda c: c["area"], reverse=True)[:2]
components = sorted(components, key=lambda c: c["cx"])  # 再排成左、右

# 畫結果
result = cv2.cvtColor(th_in_roi, cv2.COLOR_GRAY2BGR)

for idx, comp in enumerate(components):
    cx, cy = int(comp["cx"]), int(comp["cy"])
    cv2.circle(result, (cx, cy), 3, (0, 0, 255), -1)
    cv2.putText(result, f"({cx},{cy})", (cx+5, cy-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0,255,0), 1)
    print(f"group {idx+1}: center = ({cx}, {cy}), area = {comp['area']}")

cv2_imshow(result)
im_drawCircle = im.copy()

for idx, comp in enumerate(components):
    cx, cy = int(comp["cx"]), int(comp["cy"])
    cv2.circle(im_drawCircle, (cx+x1, cy+y1), int(box_height/20), (0, 0, 255), -1)
    cv2.putText(im_drawCircle, f"({cx+x1},{cy+y1})", (cx+x1+5, cy+y1-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0,255,0), 1)
    print(f"group {idx+1}: center = ({cx+x1}, {cy+y1}), area = {comp['area']}")
cv2_imshow(im_drawCircle)
